# OpenBMI (Lee2019_MI): SSM-DAN, ablations and baselines: `bft`

**This notebook runs one experiment: BFT (test-time-adaptation baseline)** (`--method bft`).
Other experiments: the other notebooks in `notebooks/openbmi/`; all of them write to the same `results/openbmi/` layout, so steps 7-8 compare every run found there.

Chronological leave-one-subject-out on OpenBMI: 54 subjects, 62 channels, 2 classes (left hand, right hand). In each fold the 53 other subjects are labeled sources (session 1 for training, session 2 for checkpoint selection); the target's unlabeled session 1 is used in Stage 1 and its session 2 for IM-TTA and evaluation.

The baselines are not in the paper on OpenBMI; they get the same 100 Hz input as SSM-DAN and their BCI IV settings (README, *Baselines and credits*).

Runs on Kaggle (GPU T4 x2, **Internet on**), on Colab, or on any machine with a GPU.

**Data** needs no preparation: the paper's preprocessed files (CAR) are part of the repository, `data/openbmi/`, and come with the clone in step 2. Only the `no_car` variants need files that are not there; step 4 links them from attached inputs (e.g. an earlier output) or builds them from the raw recordings. The expected files and folders are described in `data/README.md`.

1. Settings: experiments, subjects, seed, GPUs
2. Get the code
3. Python environment (Python 3.10 with the pinned packages)
4. Data: files already in `data/` are used; only missing ones are obtained
5. Train
6. Results of the runs, subject by subject
7. Comparison table (mean +- SD, kappa, ITR, Wilcoxon + Holm)
8. Figures: per-subject accuracy and mean confusion matrices

Every number printed or plotted is read from the runs' own output, `results/openbmi/<run>/seed<k>/`
(Kaggle: `/kaggle/working`, Colab: `/content`). If the session ends before every fold is finished, run the
notebook again with the earlier output visible (Kaggle: *Add Input -> Your Work -> Notebook Output*; Colab:
results in the mounted Drive): finished folds are copied over and skipped.

## 1. Settings

In [ ]:
DATASET = 'openbmi'
EXPERIMENTS = [('bft', '')]   # BFT (test-time-adaptation baseline)
SUBJECTS = 'all'        # 'all' | '3' | '1-4' | '1,5,9'
SEED = 0
GPUS = [0, 1]           # Kaggle T4 x2; one GPU: [0]
MODE = 'paper'          # 'paper': folds launched as in the reported runs; 'per_fold' | 'one_process'
OVERRIDES = []          # e.g. ['max_epochs=2'] for a quick check
WORKERS = 3             # OpenBMI subjects built at the same time, if any must be built from raw data
TIME_BUDGET_H = None    # None: 11.5 h on Kaggle/Colab, no limit elsewhere
BRANCH = 'main'

## 2. Get the code

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = Path('/tmp/ssm-dan')                 # outside the saved output
URLS = ['https://github.com/CuongDM1806/ssm-dan.git',
        'https://github.com/CuongDM1806/tcformer-test.git']   # former name of the repository
for url in URLS:
    if (REPO / '.git').exists():
        break
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, str(REPO)],
                   env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--oneline'], check=True)
sys.path.insert(0, str(REPO))
from ssm_dan import launch, results
from ssm_dan import config as cfg
from ssm_dan.data import SUBJECT_IDS

platform = launch.detect()                  # kaggle | colab | local
RESULTS = platform.results
CONFIGS = {}
for method, variant in EXPERIMENTS:
    variants = [v.strip() for v in variant.split(',') if v.strip()]
    config = cfg.resolve(method, DATASET, variants, OVERRIDES, seed=SEED)
    mode = launch.paper_process_mode(DATASET, variants) if MODE == 'paper' else MODE
    CONFIGS[config['name']] = dict(method=method, variant=variant, mode=mode, config=config)
TARGETS = cfg.parse_subjects(SUBJECTS, SUBJECT_IDS[DATASET])
print('platform:', platform.name, '| results:', RESULTS)
for name, run in CONFIGS.items():
    print(f'  {name:32s} mode {run["mode"]}')

## 3. Python environment

In [ ]:
PYTHON = launch.python_for_training(          # creates a Python 3.10 environment once if needed
    platform, ['transformers==4.48.3'] if any(m == 'eegencoder' for m, _ in EXPERIMENTS) else [])
env = launch.environment(platform)
GPUS = GPUS[:max(launch.available_gpus() or 1, 1)]
print('python:', PYTHON, '| GPUs used:', GPUS)

## 4. Data

Every experiment needs the 54 subjects preprocessed with its common-average-reference setting: one
`openbmi_SXX.npz` per subject (~20 MB, both sessions) in `data/openbmi/` (CAR, as in the paper) or
`data/openbmi_nocar/` (`no_car` variants). Layout and file format: `data/README.md`.

1. Files already in `data/` are used as they are; with CAR (the paper), all 54 come with the repository.
2. Missing files found in the attached inputs (Kaggle Datasets, earlier outputs) are linked into `data/`.
3. Subjects still missing (e.g. without CAR) are downloaded from the original source (~1.1 GB per subject) and
   preprocessed, `WORKERS` subjects at a time (a few hours for all 54). On Kaggle they are also saved to the
   output (`data/openbmi_nocar/`); attach that output next time.

In [ ]:
for car in sorted({run['config']['preprocessing'].get('common_average_reference', True) for run in CONFIGS.values()}):
    launch.ensure_data(DATASET, no_car=not car, workers=WORKERS, platform=platform, env=env, python=PYTHON)

## 5. Train

The OpenBMI runs train one fold per GPU at a time (one process per fold, as in the reported runs), so the experiments run one after the other over all GPUs. 54 folds take more than one Kaggle session: run the notebook again to continue. Progress is printed with its GPU and written to `results/openbmi/<run>/seed<k>/logs/`. Each
experiment gets the time left, so no fold is started that would not finish before the session ends.

In [ ]:
import queue, threading, time

budget = TIME_BUDGET_H if TIME_BUDGET_H is not None else (11.5 if platform.name in ('kaggle', 'colab') else None)
started = time.monotonic()
failed = {}

def hours_left():
    return None if budget is None else budget - (time.monotonic() - started) / 3600

def train(name, gpus):
    run, left = CONFIGS[name], hours_left()
    if left is not None and left < 0.5:
        print(f'{name} not started: {left:.1f} h left', flush=True)
        return
    try:
        launch.run(run['method'], DATASET, variant=run['variant'], subjects=SUBJECTS, seed=SEED, gpus=gpus,
                   overrides=OVERRIDES, mode=run['mode'], time_budget_h=left, platform=platform)
    except Exception as error:              # keep the other experiments going
        failed[name] = repr(error)
        print(f'{name} FAILED: {error!r}', flush=True)

# One-process experiments: one per GPU at a time.
todo = queue.Queue()
for name, run in CONFIGS.items():
    if run['mode'] == 'one_process':
        todo.put(name)

def gpu_worker(gpu):
    while True:
        try:
            name = todo.get_nowait()
        except queue.Empty:
            return
        train(name, [gpu])

threads = []
for gpu in GPUS:
    if todo.empty():
        break
    threads.append(threading.Thread(target=gpu_worker, args=(gpu,)))
    threads[-1].start()
    time.sleep(60 if len(GPUS) > 1 else 0)  # stagger the data loading of parallel runs
for thread in threads:
    thread.join()

# Per-fold experiments: one after the other, one fold per GPU at a time.
for name, run in CONFIGS.items():
    if run['mode'] == 'per_fold':
        train(name, GPUS)

print(f'Finished in {(time.monotonic() - started) / 3600:.2f} h.', f'Failed: {failed}' if failed else '')

## 6. Results of the runs

Read from each run's `subjects/SXX.json`: accuracy (%) on session 2 of every target subject; for SSM-DAN
also `no TTA`, the same weights before IM-TTA (the Stage-1 UDA model). Below: mean and population
SD over the finished subjects, mean Cohen's kappa, training time per fold and the number of finished folds.

In [ ]:
import numpy as np

def run_records(name):
    return {r['subject_id']: r for r in results.load_subjects(RESULTS / DATASET / name / f'seed{SEED}')}

RECORDS = {name: run_records(name) for name in CONFIGS}
columns = []                                  # (header, run, metric)
for name, records in RECORDS.items():
    columns.append((name, name, 'test_acc'))
    for metric, label in (('no_tta_acc', 'no TTA'), ('all_sessions_acc', 'all sessions')):
        if any(metric in r for r in records.values()):
            columns.append((f'{name} | {label}', name, metric))

def cell(value, width):
    return f'{value:>{width}}'

widths = [max(len(header), 8) for header, _, _ in columns]
print(cell('subject', 8), *[cell(h, w) for (h, _, _), w in zip(columns, widths)])
for subject in TARGETS:
    values = [RECORDS[run].get(subject, {}).get(metric) for _, run, metric in columns]
    print(cell(f'S{subject}', 8), *[cell('-' if v is None else f'{v * 100:.2f}', w) for v, w in zip(values, widths)])

def summary(label, function):
    out = []
    for (_, run, metric), w in zip(columns, widths):
        values = [r[metric] for s, r in RECORDS[run].items() if s in TARGETS and metric in r]
        out.append(cell(function(values, run, metric) if values else '-', w))
    print(cell(label, 8), *out)

summary('mean', lambda v, run, metric: f'{np.mean(v) * 100:.2f}')
summary('SD', lambda v, run, metric: f'{np.std(v) * 100:.2f}')
summary('kappa', lambda v, run, metric: f'{np.mean([RECORDS[run][s][metric.replace("acc", "kappa")] for s in RECORDS[run] if s in TARGETS]):.3f}')
summary('min/fold', lambda v, run, metric: f'{np.nanmean([r.get("train_time_min", np.nan) for r in RECORDS[run].values()]):.1f}')
summary('folds', lambda v, run, metric: f'{len(v)}/{len(TARGETS)}')

## 7. Comparison table

All OpenBMI (Lee2019_MI) runs of seed `SEED` under `results/` (also those of earlier sessions), on the subjects every run
has finished: accuracy per subject, mean +- SD (population SD), kappa, ITR and the two-sided Wilcoxon
signed-rank test against `REFERENCE` with Holm's correction. Written to `results/tables/openbmi.csv`.

In [ ]:
REFERENCE = 'ssm_dan'
ONLY = ''               # comma-separated runs; '' = every run
METRIC = 'test_acc'     # test_acc | no_tta_acc | all_sessions_acc

command = [str(PYTHON), '-m', 'ssm_dan.tables', '--results', str(RESULTS), '--dataset', DATASET,
           '--reference', REFERENCE, '--metric', METRIC, '--seed', str(SEED)]
subprocess.run(command + (['--runs', ONLY] if ONLY else []), cwd=REPO, env=env, check=True)

## 8. Figures

Per-subject accuracy of the runs, and each run's mean confusion matrix (every subject's matrix normalized by
the trials of each true class, then averaged over subjects) with its macro-F1 (mean over subjects).
Saved to `results/openbmi/figures/`.

In [ ]:
import matplotlib.pyplot as plt

FIGURES = RESULTS / DATASET / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
finished = {name: records for name, records in RECORDS.items() if records}

if finished:
    n_classes = len(next(iter(next(iter(finished.values())).values()))['class_names'])
    fig, ax = plt.subplots(figsize=(min(18, max(10, 0.3 * len(TARGETS) * max(len(finished), 2))), 3.8))
    width = 0.8 / len(finished)
    for k, (name, records) in enumerate(finished.items()):
        subjects = [s for s in TARGETS if s in records]
        xs = [TARGETS.index(s) + (k - (len(finished) - 1) / 2) * width for s in subjects]
        ax.bar(xs, [records[s]['test_acc'] * 100 for s in subjects], width, label=name)
    ax.axhline(100 / n_classes, color='gray', lw=0.8, ls='--', label='chance')
    ax.set(xticks=range(len(TARGETS)), ylabel='accuracy (%)', ylim=(0, 100), xlim=(-0.6, len(TARGETS) - 0.4))
    ax.set_xticklabels([f'S{s}' for s in TARGETS], fontsize=8 if len(TARGETS) <= 12 else 6, rotation=0 if len(TARGETS) <= 12 else 90)
    ax.set_title(f'{DATASET}, seed {SEED}', loc='left')
    ax.legend(fontsize=8, ncol=len(finished) + 1, loc='lower right', bbox_to_anchor=(1, 1), frameon=False)
    fig.tight_layout()
    fig.savefig(FIGURES / f'per_subject_accuracy_seed{SEED}.png', dpi=150)
    plt.show()

    fig, axes = plt.subplots(1, len(finished), figsize=(3.6 * len(finished), 3.5), squeeze=False)
    for ax, (name, records) in zip(axes[0], finished.items()):
        classes = next(iter(records.values()))['class_names']
        counts = np.array([r['confusion_counts'] for r in records.values()], dtype=float)
        recall = counts / counts.sum(axis=2, keepdims=True).clip(min=1)
        mean = recall.mean(axis=0) * 100
        precision = np.diagonal(counts, axis1=1, axis2=2) / counts.sum(axis=1).clip(min=1)
        diagonal = np.diagonal(recall, axis1=1, axis2=2)
        macro_f1 = np.mean(2 * precision * diagonal / (precision + diagonal).clip(min=1e-12), axis=1).mean()
        ax.imshow(mean, cmap='Blues', vmin=0, vmax=100)
        for i in range(len(classes)):
            for j in range(len(classes)):
                ax.text(j, i, f'{mean[i, j]:.1f}', ha='center', va='center', fontsize=8,
                        color='white' if mean[i, j] > 50 else 'black')
        labels = [c.replace('_', ' ') for c in classes]
        ax.set(xticks=range(len(classes)), yticks=range(len(classes)), xticklabels=labels, yticklabels=labels,
               xlabel='predicted', ylabel='true', title=f'{name}\nmacro-F1 {macro_f1 * 100:.2f}')
        ax.tick_params(axis='x', labelrotation=45, labelsize=8)
        ax.tick_params(axis='y', labelsize=8)
    fig.tight_layout()
    fig.savefig(FIGURES / f'mean_confusion_seed{SEED}.png', dpi=150)
    plt.show()
else:
    print('No finished folds yet.')

Per-fold confusion matrices and learning curves are in each run's `figures/`, the trained checkpoints in
`checkpoints/`, the resolved configuration in `config.yaml`. On Kaggle, save a version of the notebook to keep
`/kaggle/working`; attach that output to the next session to continue unfinished runs.